## Problem Statement : Hospital Patient Data Analysis

## Context:
#### A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### 1. Load the patient dataset and show summary with info().

In [2]:
P_data = pd.read_csv("Patient_data.csv")
P_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


#### 2. Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [3]:
billing_df = P_data[['PatientID','Department','Doctor','BillAmount']]
billing_df.head()

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


#### 3. Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [4]:
P_data = P_data.drop(['ReceptionistID', 'CheckInTime'],axis=1)
P_data.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


#### 4. Use groupby to find total bill amount per department.

In [8]:
dep_bill = billing_df.groupby('Department')['BillAmount'].sum()
dep_bill

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

#### 5. Remove duplicate patient records based on PatientID.

In [9]:
P_data = P_data.drop_duplicates(subset='PatientID')
P_data.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


#### 6. Fill missing BillAmount values with the mean bill amount.

In [10]:
billing_df['BillAmount'] = billing_df['BillAmount'].fillna(billing_df['BillAmount'].mean())

#### 7. Merge the billing dataset with patient dataset on PatientID.

In [16]:
billing_data = pd.read_csv('Billing_Data.csv')
merge_data =  pd.merge(P_data,billing_data,on='PatientID')
merge_data.head()

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,2000,3000
1,102,Bob,Neurology,Dr. John,NaN,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.0,3000,3200
4,105,Eva,Dermatology,Dr. Rose,NaN,1000,4000


#### 8. Concatenate an additional DataFrame that contains new patients for the current week      (row-wise).

In [17]:
new_patient = pd.DataFrame({
    'PatientID':[301,302],
    'Department':['Cardiology','Orthopedic'],
    'Doctor':['Dr Smith','Dr Lee'],
    'BillAmount':[5000,4500]
})

In [21]:
updated_data = pd.concat([P_data,new_patient],axis=0,ignore_index=True)
updated_data.tail()

,PatientID,Name,Department,Doctor,BillAmount
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,301,NaN,Cardiology,Dr Smith,5000.0
6,302,NaN,Orthopedic,Dr Lee,4500.0


#### 9. Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [22]:
## updated dat with extra columns
extra_columns = pd.DataFrame({
    'InsuranceCovered':[1000,1200,900,800,950],
    'FinalAmount':[4000,3800,4100,4200,3500]
})
updated_data = pd.concat([P_data,extra_columns],axis=1)
updated_data.tail()

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,1000,4000
1,102,Bob,Neurology,Dr. John,NaN,1200,3800
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,900,4100
3,104,David,Cardiology,Dr. Smith,6200.0,800,4200
4,105,Eva,Dermatology,Dr. Rose,NaN,950,3500


### Expected Outcomes :
#### The final cleaned dataset contains accurate patient and billing information with missing values handled and duplicate records removed. The patient and billing datasets were merged using PatientID to create a unified dataset. This dataset can be used for further analysis such as department-wise revenue and doctor performance evaluation.